In [0]:
"""
STEP 02: Create and call endpoint with dataset
"""

In [0]:
from databricks.sdk import WorkspaceClient
from loguru import logger
from openai import OpenAI
from pyspark.sql import SparkSession

from mushroom_data_preprocessing.config import settings

In [0]:
w = WorkspaceClient()
spark = SparkSession.builder.getOrCreate()

settings.configure("dev")

host = settings.get("host")
token = w.tokens.create(lifetime_seconds=1200).token_value

client = OpenAI(api_key=token, base_url=f"{host.rstrip('/')}/serving-endpoints")

model_name = "databricks-claude-sonnet-4-6"
catalog = settings.dev.get("catalog")
schema = settings.dev.get("schema")
table_name = "mushroom_metadata"

query = f"""
  SELECT *
  FROM {catalog}.{schema}.{table_name}
  """

In [0]:
query_df = spark.sql(query)
all_text = query_df.toPandas().to_markdown(index=False)
logger.info("Query success: connected to data")

2026-04-07 19:45:17.941 | INFO     | __main__:<module>:3 - Query success: connected to data


In [0]:
response = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant with access to "
                "a mushroom dataset, which is in German and which "
                "you will have to translate."
                "Questions will be posed to you in English."
                "If the data is insufficient to answer, say you don't know."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Mushroom metadata:\n\n{all_text}\n\n"
                "How many mushrooms are poisonous versus edible?"
            ),
        },
    ],
    max_tokens=500,
    temperature=0.7,
)

logger.info("Response:")
logger.info(response.choices[0].message.content)
logger.info(f"Tokens used: {response.usage.total_tokens}")
logger.info(f"Input tokens: {response.usage.prompt_tokens}")
logger.info(f"Output tokens: {response.usage.completion_tokens}")

2026-04-07 19:45:31.385 | INFO     | __main__:<module>:25 - Response:
2026-04-07 19:45:31.386 | INFO     | __main__:<module>:26 - Based on the data provided, here is the breakdown:

- **ESSBAR (Edible):** 20 mushrooms
- **UNGENIESSBAR (Inedible/Not edible):** 11 mushrooms

Note: None of the mushrooms in this dataset are explicitly labeled as **GIFTIG (Poisonous/Toxic)**. The "UNGENIESSBAR" category means they are **inedible** (not suitable for eating), but not necessarily poisonous in the toxic sense.
2026-04-07 19:45:31.386 | INFO     | __main__:<module>:27 - Tokens used: 332335
2026-04-07 19:45:31.387 | INFO     | __main__:<module>:28 - Input tokens: 332217
2026-04-07 19:45:31.387 | INFO     | __main__:<module>:29 - Output tokens: 118
